[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/normals/normals.ipynb)

# Surface Normals and Reflections

This example computes how surface normals, lines, and intersecting planes transform across a mirror when they carry opposite orientations. A surface normal specifies either a chosen side of a surface, as used in half-space collision tests, or the circulation of ordered vertices around a face, as used in back-face culling, and reflecting a triangulated surface separates the two.

In 3D projective geometric algebra, a boundary plane carries extrinsic orientation whereas joining ordered vertices with `triangle[0] & triangle[1] & triangle[2]` produces a plane with intrinsic orientation, and the reflection operator `mirror >> x` transforms both consistently. Along a line lying in the mirror, the point join `ends[0] & ends[1]` stays fixed under `mirror >> line` whereas the plain sandwich `mirror * line * mirror.inverse()` reverses it, and the plane meet `planes[0] ^ planes[1]` negates when reflection swaps the two crossing planes, following Leo Dorst in [Projective duality encodes complementary orientations in geometric algebras](https://doi.org/10.1002/mma.9754).

In homogeneous-matrix and Gibbs–Heaviside vector notation, a boundary plane $\mathbf{n}^\top \mathbf{x} + d = 0$ transforms as a covector $(M^{-1})^\top (\mathbf{n}, d)$ with polar normal $(A^{-1})^\top \mathbf{n}$, whereas a vertex-winding cross product $\mathbf{n}_w = (\mathbf{p}_1 - \mathbf{p}_0) \times (\mathbf{p}_2 - \mathbf{p}_0)$ transforms as an axial pseudovector $(\det A)(A^{-1})^\top \mathbf{n}_w$ carrying an extra $\det A = -1$ sign factor under reflection. Likewise, a 3D line splits into Plücker coordinates $(\mathbf{d}, \mathbf{m}) = (\mathbf{p}_1 - \mathbf{p}_0,\, \mathbf{p}_0 \times \mathbf{p}_1)$ whose polar direction $A\mathbf{d}$ and axial moment $(\det A)(A^{-1})^\top \mathbf{m}$ transform by different matrix rules depending on whether the line is formed from two points or two planes.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from examples.orientation.normals import render
from examples.util.animation import save_figure
from numga import NumpyContext, stack
from numga.algebras import PGA3D

# + extrinsic, - intrinsic
ga = PGA3D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                                                  # Antivector-
Plane = ga.gatype.vector()                                                      # Vector+
Surface = ga.gatype(Point & Point & Point)                                      # Antitrivector-
Spear = ga.gatype(Point & Point)                                                # Antibivector-
Axis = ga.gatype(Plane ^ Plane)                                                 # Bivector+

## Side and winding under reflection

A horizontal triangle sits beside a vertical mirror with its positive boundary side above the plane and its vertices ordered counterclockwise as seen from above. Before reflection both orientations point upward, and reflecting across the vertical mirror carries the boundary plane and the ordered vertices by the same `mirror >> x` sandwich.

In [ ]:
triangle = mv.point([[-2.4, -0.65, 0], [-0.9, -0.45, 0], [-1.8, 0.8, 0]])       # [3] Point-
side = mv.z                                                                     # [] Plane+
mirror = mv.x                                                                   # [] Plane+

In [ ]:
# Joining the ordered vertices gives the triangle's plane its intrinsic winding orientation.
winding = (triangle[0] & triangle[1] & triangle[2]).normalized()                # [] Surface-

# The same reflection acts on points, the boundary side, and the winding plane alike.
vertices = stack([triangle, mirror >> triangle])                                # [2, 3] Point-
sides = stack([side, mirror >> side])                                           # [2] Plane+
windings = stack([winding, mirror >> winding])                                  # [2] Surface-
save_figure(render.normals_figure(vertices, sides, windings, mirror), "normal_orientation");

The translucent blue-grey sheet marks the mirror. In the left panel, teal arrows mark the boundary plane's positive side, which stays above both triangles. In the right panel, orange edge arrows follow the vertex order and the orange normal arrow displays that circulation using the camera's ambient handedness, so the mirrored winding points downward while the positive side stays upward. Joining the reflected vertices in order produces the exact same downward winding plane as reflecting the original winding plane.

## A line lying in the mirror

Two points lying in the mirror plane are each fixed by reflection, so the directed line joining them is fixed as well. Reflecting that joined line with `mirror >> line` preserves its direction along the mirror, whereas the orientation-blind sandwich `mirror * line * mirror.inverse()` applies the same sign to every bivector and reverses the line even though neither endpoint moves.

In [ ]:
ends = mv.point([[0, -1, 0.2], [0, 1, 0.8]])                                    # [2] Point-

In [ ]:
line = ends[0] & ends[1]                                                        # [] Spear-
# The plain sandwich applies one grade sign regardless of orientation, reversing the fixed line.
lines = stack([line, mirror >> line, mirror * line * mirror.inverse()])         # [3] Spear-
save_figure(render.line_in_mirror_figure(ends, lines, mirror), "line_in_mirror");

Both dark spheres lie inside the mirror sheet. The left panel shows the joined line directed from the lower point to the upper point, the middle panel shows `mirror >> line` pointing the same way, and the right panel shows the plain sandwich reversing the arrow between the two unmoved points.

## Meets of planes and joins of points in the mirror

Two planes inclined at equal and opposite angles to the mirror intersect along a line inside the mirror and swap places under reflection, while two points on that intersection line stay fixed. Because the outer product is antisymmetric, intersecting the swapped mirror-image planes in `plane_pairs[:, 0] ^ plane_pairs[:, 1]` negates their meet, whereas joining the fixed points in `point_pairs[:, 0] & point_pairs[:, 1]` leaves their join unchanged.

In [ ]:
crossing_planes = stack([mv.x + mv.y, mv.y - mv.x])                             # [2] Plane+
line_points = mv.point([[0, 0, -0.5], [0, 0, 0.5]])                             # [2] Point-

In [ ]:
# Reflect both pairs across the mirror and form the plane meet and point join in each scene.
plane_pairs = stack([crossing_planes, mirror >> crossing_planes])               # [2, 2] Plane+
point_pairs = stack([line_points, mirror >> line_points])                       # [2, 2] Point-
meets = plane_pairs[:, 0] ^ plane_pairs[:, 1]                                   # [2] Axis+
joins = point_pairs[:, 0] & point_pairs[:, 1]                                   # [2] Spear-
save_figure(render.pairs_in_mirror_figure(plane_pairs, point_pairs, meets, joins, mirror), "pairs_in_mirror");

The left column shows the original pair and the right column shows its mirror image. In the top row, the red and blue points on the mirror line stay in place so their join keeps its sign, whereas in the bottom row the red and blue planes swap across the mirror sheet so their meet reverses sign.

In [ ]:
# checks
witness = vertices[0].mean(axis=0) + mv.z.dual()                                # [] Point-
reflected_witness = mirror >> witness                                           # [] Point-
rebuilt_winding = (vertices[1, 0] & vertices[1, 1] & vertices[1, 2]).normalized()  # [] Surface-

assert ((sides[0] & witness) - 1).abs() < 1e-6
assert ((sides[1] & reflected_witness) - 1).abs() < 1e-6
assert ((windings[0] & witness) + (windings[1] & reflected_witness)).scalar_norm_squared() < 1e-12
rebuilt_diff = rebuilt_winding - windings[1]                                    # [] Surface-
assert rebuilt_diff.scalar_norm_squared() + rebuilt_diff.dual().scalar_norm_squared() < 1e-12

reflected_line_diff = lines[1] - lines[0]                                       # [] Spear-
sandwich_line_sum = lines[2] + lines[0]                                         # [] Spear-
assert reflected_line_diff.scalar_norm_squared() + reflected_line_diff.dual().scalar_norm_squared() < 1e-12
assert sandwich_line_sum.scalar_norm_squared() + sandwich_line_sum.dual().scalar_norm_squared() < 1e-12

swapped_diff = plane_pairs[1] - plane_pairs[0, ::-1]                            # [2] Plane+
fixed_ends_diff = point_pairs[1] - point_pairs[0]                               # [2] Point-
meets_sum = meets[1] + meets[0]                                                 # [] Axis+
joins_diff = joins[1] - joins[0]                                                # [] Spear-
assert ((swapped_diff.scalar_norm_squared() + swapped_diff.dual().scalar_norm_squared()) < 1e-12).all()
assert ((fixed_ends_diff.scalar_norm_squared() + fixed_ends_diff.dual().scalar_norm_squared()) < 1e-12).all()
assert meets_sum.scalar_norm_squared() + meets_sum.dual().scalar_norm_squared() < 1e-12
assert joins_diff.scalar_norm_squared() + joins_diff.dual().scalar_norm_squared() < 1e-12